## Image denoise

#### <p style="text-align: right;"> &#9989; **put your name here** </p>

**In this file, we will build a low-pass filter using FFT.**

- Load a SEM image of a graphite electrode.
- `https://raw.githubusercontent.com/huichiayu/cmse_202_802/main/MSE590/Graphite_FIBSEM.jpg`
- Plot the image.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# -----------------------------
# 1. Load original image
# -----------------------------
img = Image.open("Graphite_FIBSEM.jpg").convert("L")
I = np.array(img, dtype=float)

# Normalize to 0–1
I_norm = I / 255.0

plt.figure(figsize=(6, 6))

plt.imshow(I_norm, cmap="gray")
plt.title("Original")
plt.axis("off")

print(I_norm.shape)

- Use FFT to plot the spectra. 

In [ ]:
# -----------------------------
# 2. FFT of noisy image
# -----------------------------
F = np.fft.fft2(I_norm)
F_shift = np.fft.fftshift(F)


plt.figure(figsize=(6, 6))

plt.imshow(np.log(1+ np.abs(F_shift)), cmap="gray")
plt.title("FFT spectrum")
plt.colorbar()
plt.axis("off")

- Create a mask, where items in radius less than R will pass.

In [ ]:
# -----------------------------
# 3. Low-pass filter
# -----------------------------
rows, cols = I_norm.shape
crow, ccol = rows // 2, cols // 2

R = 20   # cutoff radius; tune this

y, x = np.ogrid[:rows, :cols]
distance = np.sqrt((x - ccol)**2 + (y - crow)**2)

mask = distance <= R

F_filtered = F_shift * mask


- Use inverse FFT to convert the spectra (frequency domain) back to image (space domain).

In [ ]:
# -----------------------------
# 4. Inverse FFT
# -----------------------------
F_ishift = np.fft.ifftshift(F_filtered)
I_denoised = np.fft.ifft2(F_ishift)
I_denoised = np.real(I_denoised)

I_denoised = np.clip(I_denoised, 0, 1)

plt.figure(figsize=(6, 6))
plt.imshow(I_denoised, cmap="gray")
plt.title("FFT denoised")
plt.axis("off")


&#9989; Do This -- Answer these questions.
- Why `ifftshist` is need in the cell above?
- Does the denoise filter works?

---
### Other filters or denoising methods.

- Guassian smooth

In [ ]:
from scipy.ndimage import gaussian_filter

denoised_1 = gaussian_filter(I_norm, sigma=1.0)

plt.figure(figsize=(6, 6))
plt.imshow(denoised_1, cmap="gray")
plt.title("Guassian denoised")
plt.axis("off")

In [ ]:
from scipy.ndimage import median_filter

denoised_2 = median_filter(I_norm, size=3)

plt.figure(figsize=(6, 6))
plt.imshow(denoised_2, cmap="gray")
plt.title("Median filter")
plt.axis("off")

&#9989; Do This -- Answer this question.
- What is a median filter?

In [ ]:
from skimage.restoration import denoise_nl_means, estimate_sigma
import numpy as np

sigma_est = np.mean(estimate_sigma(I_norm, channel_axis=None))

denoised_3 = denoise_nl_means(
    I_norm,
    h=1.15 * sigma_est,
    fast_mode=True,
    patch_size=5,
    patch_distance=6
)

plt.figure(figsize=(6, 6))
plt.imshow(denoised_3, cmap="gray")
plt.title("Non-local means")
plt.axis("off")

In [ ]:
from skimage.restoration import denoise_bilateral

denoised_4 = denoise_bilateral(I_norm, sigma_color=0.05, sigma_spatial=3)

plt.figure(figsize=(6, 6))
plt.imshow(denoised_3, cmap="gray")
plt.title("Bilateral filter")
plt.axis("off")

In [ ]:
from skimage.restoration import denoise_tv_chambolle

denoised_5 = denoise_tv_chambolle(I_norm, weight=0.1)

plt.figure(figsize=(6, 6))
plt.imshow(denoised_3, cmap="gray")
plt.title("Total Variation")
plt.axis("off")

&#9989; Do This -- Answer thess questions.
- In your own opnion, which of these three methods - Non-local means, Bilateral filter, and Total Variation, works best?
- Do a search and briefly describe how the one you choose works.

### You're done.

Please upload your file via this [**link**](https://www.dropbox.com/request/c9h5elak5brdtlrjoj5b).